# 02 — Preparación de Datos para Deep Learning
Pipeline completo: filtrado → binarización → TF-IDF → split train/test

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataset import load_and_filter, binarize, build_tag_features, split_by_user
from config import MIN_INTERACTIONS, TAG_FEATURES

print('Librerías cargadas')

## Paso 1: Filtrado mínimo de interacciones

In [ ]:
raw = pd.read_csv('../data/lastfm/user_artists.dat', sep='\t', encoding='latin-1')
print(f'Antes del filtro: {len(raw):,} interacciones, {raw.userID.nunique():,} usuarios, {raw.artistID.nunique():,} artistas')

filtered = load_and_filter(min_interactions=MIN_INTERACTIONS)
print(f'Después del filtro (>={MIN_INTERACTIONS}): {len(filtered):,} interacciones, {filtered.userID.nunique():,} usuarios, {filtered.artistID.nunique():,} artistas')

## Paso 2: Binarización (feedback implícito)

In [ ]:
df = binarize(filtered)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['weight_log'].values, bins=50, alpha=0.7, color='steelblue')
axes[0].axvline(df['weight_log'].median(), color='red', linestyle='--', label='Mediana')
axes[0].set_title('Distribución de log1p(weight)'); axes[0].legend()

label_counts = df['label'].value_counts()
axes[1].bar(['Dislike (0)', 'Like (1)'], [label_counts.get(0,0), label_counts.get(1,0)],
            color=['coral', 'seagreen'], alpha=0.8)
axes[1].set_title('Distribución de labels binarios')
plt.tight_layout(); plt.show()

print(f'Labels: {label_counts.to_dict()}')

## Paso 3: Features de tags (TF-IDF)

In [ ]:
artist_ids = sorted(df['artistID'].unique())
tag_matrix, vectorizer = build_tag_features(artist_ids)

print(f'Forma de la matriz de tags: {tag_matrix.shape}  (artistas × {TAG_FEATURES} features)')
print(f'Artistas sin tags (vector cero): {(tag_matrix.sum(axis=1) == 0).sum()}')
print(f'Top 10 features TF-IDF: {vectorizer.get_feature_names_out()[:10]}')

## Paso 4: Split train/test por usuario (80/20)

In [ ]:
train_df, test_df = split_by_user(df)

print(f'Train: {len(train_df):,} interacciones ({train_df.userID.nunique()} usuarios)')
print(f'Test:  {len(test_df):,}  interacciones ({test_df.userID.nunique()}  usuarios)')

# Verificar sin data leakage
train_keys = set(zip(train_df.userID, train_df.artistID))
test_keys  = set(zip(test_df.userID,  test_df.artistID))
overlap    = train_keys & test_keys
assert len(overlap) == 0, f'Data leakage detectado: {len(overlap)} pares en ambos sets'
print('✓ Sin data leakage entre train y test')

## Paso 5: Dataset PyTorch con negative sampling

In [ ]:
from dataset import LastFMDataset
from config import NEG_RATIO

user_ids   = sorted(df['userID'].unique())
artist_ids = sorted(df['artistID'].unique())
user_to_idx   = {u: i for i, u in enumerate(user_ids)}
artist_to_idx = {a: i for i, a in enumerate(artist_ids)}

train_dataset = LastFMDataset(train_df, tag_matrix, user_to_idx, artist_to_idx, neg_ratio=NEG_RATIO)

print(f'Pares de entrenamiento: {len(train_dataset):,}')
print(f'  Positivos: {(train_df.label == 1).sum():,}')
print(f'  Negativos: {len(train_dataset) - (train_df.label == 1).sum():,} (ratio {NEG_RATIO}:1)')

# Muestra un elemento
u_idx, a_idx, tags, label = train_dataset[0]
print(f'\nEjemplo de elemento:')
print(f'  user_idx={u_idx.item()}, artist_idx={a_idx.item()}, tag_shape={tags.shape}, label={label.item()}')

## Resumen del pipeline
```
user_artists.dat → filter(≥5) → log1p + binarize → split(80/20)
user_taggedartists.dat → group_by_artist → TF-IDF(50) → tag_matrix
→ LastFMDataset → negative_sampling(4:1) → DataLoader
```
Todos los objetos (`user_to_idx`, `artist_to_idx`, `tag_matrix`, `train_dataset`) se usan en `03_modelo_ncf.ipynb`.